# 01 - Pull Statcast data

**Goal:** download pitch-level Statcast data for each regular season in `SEASONS` (see `src/config.py`) and run basic checks.

Raw files are saved to `data/raw/` (git-ignored). Only small summary tables go to `data/processed/` so they can be committed.

In [1]:
import sys
sys.path.append("..")  # lets the notebook import from src/

import pandas as pd
from src.config import SEASONS, PROCESSED
from src.data import pull_season, load_seasons, add_park

SEASONS

[2022, 2023, 2024, 2025, 2026]

## Pull each season

The first run is slow (a full season can take 10-20+ minutes). Each season is saved as a parquet file, and seasons that already exist are skipped, so it is safe to re-run.

In [ ]:
for year in SEASONS:
    pull_season(year)

## Load and label parks

In [ ]:
df = add_park(load_seasons())
print(df.shape)
df.head()

## Check what is populated

`arm_angle` is the one to watch. If it is mostly empty in early seasons, either start the analysis window later (edit `SEASONS` in `src/config.py`) or fall back to release height (the archetype code can do this and tells you which source it used).

In [ ]:
key = ["release_speed", "release_spin_rate", "pfx_x", "pfx_z", "release_pos_x", "release_pos_z",
       "release_extension", "arm_angle", "launch_speed", "estimated_woba_using_speedangle"]
key = [c for c in key if c in df.columns]
df.groupby("game_year")[key].agg(lambda s: s.isna().mean()).round(3)

## Check parks and games

Each park should have about 81 home games per season (a few fewer if games were never made up). Spot-check anything unusual, such as a team that changed stadiums. Neutral-site games (international series) get labeled with the designated home team's park by this simple mapping. It is only a handful of games per year, but if you want to remove them, look up their `game_pk` values and add them to `NEUTRAL_SITE_GAME_PKS` in `src/data.py`.

In [ ]:
coverage = (df.groupby(["game_year", "park"])
              .agg(pitches=("pitch_type", "size"), games=("game_pk", "nunique"))
              .reset_index())
coverage.query("games != 81").sort_values("games")

In [ ]:
coverage.to_csv(PROCESSED / "data_coverage.csv", index=False)

## Commit

```
git add notebooks/01_pull_data.ipynb data/processed/data_coverage.csv
git commit -m "add data pull and coverage checks"
git push
```